实现按长度分组的匹配率、答案token困惑度、耗时聚合，再运行真实本地模型评估。 本Notebook在Python环境运行，GitHub Pages不执行PyTorch或真实模型。先完成TODO，再运行验证；最后的参考实现可按需查看。


In [ ]:
%pip install torch transformers


In [ ]:
"""长上下文评估工具。安装: pip install torch transformers
默认--self-test只验证评估代码，不测模型能力。
真实评估: python 12-long-context-evaluation.py --model /path/to/local-causal-lm --samples evaluation.jsonl
JSONL每行: {"prompt":"...","answer":"...","length_bucket":"short/long"}
使用相同模型和数据比较不同设置；本工具不会替你修改模型RoPE配置。
"""
import argparse
import json
import math
import time

def aggregate(rows):
    # TODO：完成本实验目标，再执行下方验证。
    raise NotImplementedError("请完成 aggregate")

def evaluate(model_path,samples_path,max_new_tokens):
    import torch
    from transformers import AutoModelForCausalLM,AutoTokenizer
    tokenizer=AutoTokenizer.from_pretrained(model_path,local_files_only=True)
    model=AutoModelForCausalLM.from_pretrained(model_path,local_files_only=True).eval()
    rows=[]
    for line in open(samples_path,encoding='utf-8'):
        item=json.loads(line);prompt=item['prompt'];answer=item['answer']
        # 分别编码后拼接，明确答案损失的token边界。
        prefix=tokenizer.encode(prompt,add_special_tokens=True)
        suffix=tokenizer.encode(answer,add_special_tokens=False)
        if not prefix or not suffix:raise ValueError('prompt和answer必须均有token')
        ids=torch.tensor([prefix+suffix]);labels=ids.clone();labels[:,:len(prefix)]=-100
        began=time.perf_counter()
        with torch.no_grad():
            loss=model(input_ids=ids,labels=labels).loss
            generated=model.generate(torch.tensor([prefix]),max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.eos_token_id)
        prediction=tokenizer.decode(generated[0,len(prefix):],skip_special_tokens=True).strip()
        rows.append({'length_bucket':item['length_bucket'],'prompt_tokens':len(prefix),'answer_tokens':len(suffix),
                     'answer_nll':float(loss)*len(suffix),'exact_match':prediction==answer.strip(),'prediction':prediction,
                     'latency_seconds':time.perf_counter()-began})
    if not rows:raise ValueError('评估集为空')
    return {'model':model_path,'rows':rows,'summary':aggregate(rows)}



完成目标函数后，运行以下验证。初始代码出现NotImplementedError是预期状态。


In [ ]:
aggregate([dict(length_bucket='self-test',exact_match=True,answer_nll=2,answer_tokens=2,latency_seconds=.1)])


参考实现：先自行完成，再运行下一格替换目标函数，对照结果。


In [ ]:
def aggregate(rows):
    groups={}
    for row in rows:groups.setdefault(row['length_bucket'],[]).append(row)
    return {key:{'samples':len(values),'exact_match':sum(v['exact_match'] for v in values)/len(values),
                 'answer_perplexity':math.exp(sum(v['answer_nll'] for v in values)/sum(v['answer_tokens'] for v in values)),
                 'mean_latency_seconds':sum(v['latency_seconds'] for v in values)/len(values)} for key,values in groups.items()}


In [ ]:
aggregate([dict(length_bucket='self-test',exact_match=True,answer_nll=2,answer_tokens=2,latency_seconds=.1)])


上面的fixture只检查评估器，不评估真实模型。真实评估需本地模型和JSONL标注集（prompt、answer、length_bucket），调用 evaluate("模型目录", "evaluation.jsonl", 32)。比较不同位置策略时分别加载对应配置，保持数据相同。耗时包含loss计算与生成。


配套数据与说明：同目录的 README.md、retrieval-evaluation.jsonl、12-make-retrieval-set.py。先用6条小样例跑通，再按模型容量生成更长评估集；每条真实prompt token数由评估器报告。
